# Serve

In [14]:
import markdown
import pandas as pd
import uuid
import json
from langchain.messages import AIMessage, HumanMessage
from knowledge.test.model_3 import agent
from fastapi import FastAPI, Form, File, UploadFile, HTTPException
from tempfile import NamedTemporaryFile
import uvicorn
from fastapi.staticfiles import StaticFiles
import shutil
import logging
from pathlib import Path
import webbrowser

CSV_PATH = './knowledge/youtube_watch_log.csv'

def md_to_html(md_source: str) -> str:
    """Converts markdown to HTML and sanitizes image paths"""
    html = markdown.markdown(md_source, extensions=['tables', 'fenced_code'])
    # Replace image src paths
    html = html.replace('src="./knowledge/thumbnails/', 'src="/thumbnails/')
    html = html.replace('src="./temp/', 'src="/uploads/')
    return html

log = logging.getLogger(f"uvicorn.{__name__}")

In [15]:
app = FastAPI()

@app.get("/api/users")
def get_available_users() -> list[int]:
    return pd.read_csv(CSV_PATH)["user_id"].unique().tolist()

@app.post("/api/chat")
async def chat(
    user_id: int = Form(...),   # multipart/form-data
    message: str = Form(...),
    thread_id: str | None = Form(None),
    image: UploadFile | None = File(None),
) -> dict:
    if not message.strip():
        raise HTTPException(400, "message cannot be empty")

    # Generate thread ID if not provided
    thread_id = thread_id or uuid.uuid4().hex

    # Single optional image as multipart file upload
    if image:
        tmp = NamedTemporaryFile(dir="./temp", delete=False, suffix=image.filename, prefix="")
        shutil.copyfileobj(image.file, tmp)
        message += "\nAttached image: " + tmp.name

    # Invoke model
    messages = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config={"configurable": {"thread_id": thread_id}},
        context={"user_id": user_id},
    ).get("messages", [])

    html_response = ""
    for msg in messages:
        log.info(f"User[{user_id}] - Thread[{thread_id}] {msg.type}: {msg.content}")
        # Obtain HTML from last message
        html_response = md_to_html(str(msg.content))

    # Stream model response
    return {"thread_id": thread_id, "html_response": html_response}

# Serve the static content
app.mount("/thumbnails", StaticFiles(directory="knowledge/thumbnails"))
app.mount("/uploads", StaticFiles(directory="temp"))
app.mount("/", StaticFiles(directory="frontend", html=True))

# Open frontend
webbrowser.open("http://127.0.0.1:8000/index.html")

# Start the server
await uvicorn.Server(uvicorn.Config(app)).serve()

INFO:     Started server process [79838]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:59000 - "GET /index.html HTTP/1.1" 200 OK
INFO:     127.0.0.1:59000 - "GET /style.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:59001 - "GET /script.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:59000 - "GET /api/users HTTP/1.1" 200 OK
INFO:     127.0.0.1:59002 - "GET /apple-touch-icon-precomposed.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:59002 - "GET /apple-touch-icon.png HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:59002 - "GET /favicon.ico HTTP/1.1" 404 Not Found


INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] human: hello
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] ai: Hello! How can I assist you today?


INFO:     127.0.0.1:59006 - "POST /api/chat HTTP/1.1" 200 OK


INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] human: hello
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] ai: Hello! How can I assist you today?
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] human: what are the last three videos I have watched
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] ai: 
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] tool: [{"video_id": "F8wJdvXK5yU", "video_title": "SkiVel", "timestamp": "2018-09-11 17:02:26", "thumbnail_url": "./knowledge/thumbnails/F8wJdvXK5yU.jpg"}, {"video_id": "U77V6z_RYpM", "video_title": "Why Small Biases Matter Over Time", "timestamp": "2018-09-11 16:59:59", "thumbnail_url": "./knowledge/thumbnails/U77V6z_RYpM.jpg"}, {"video_id": "8DUSSbJPfwU", "video_title": "The system nudges more than you think", "timestamp": "2018-09-11 16:58:59", "thumbnail_url": "./knowledge/thumbnails/8DUSSbJPfwU.jpg"}]
INFO:     User[25] - Thread[f70ac3fb595a45659bc5a0f2736de52c] ai: * 

...

INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] human: hi watt dasu da
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] ai: Could you please clarify what you mean by "watt dasu da"? Are you looking for information or assistance with something specific?
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] human: parli anche italiano?
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] ai: Sì, parlo anche italiano. Come posso aiutarti oggi?


INFO:     127.0.0.1:59136 - "POST /api/chat HTTP/1.1" 200 OK


INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] human: hi watt dasu da
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] ai: Could you please clarify what you mean by "watt dasu da"? Are you looking for information or assistance with something specific?
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] human: parli anche italiano?
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] ai: Sì, parlo anche italiano. Come posso aiutarti oggi?
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] human: mostrami gli ultimi 3 video guardati
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] ai: 
INFO:     User[25] - Thread[48fbd395cd3c49848f864c5ac24c2285] tool: [{"video_id": "F8wJdvXK5yU", "video_title": "SkiVel", "timestamp": "2018-09-11 17:02:26", "thumbnail_url": "./knowledge/thumbnails/F8wJdvXK5yU.jpg"}, {"video_id": "U77V6z_RYpM", "video_title": "Why Small Biases Matter Over Time", "timestamp": "2018-09-11 16:59:59", "thu

INFO:     127.0.0.1:59139 - "POST /api/chat HTTP/1.1" 200 OK
INFO:     127.0.0.1:59139 - "GET /thumbnails/F8wJdvXK5yU.jpg HTTP/1.1" 200 OK
INFO:     127.0.0.1:59143 - "GET /thumbnails/U77V6z_RYpM.jpg HTTP/1.1" 200 OK
INFO:     127.0.0.1:59144 - "GET /thumbnails/8DUSSbJPfwU.jpg HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [79838]
